Tính chất 1: 
$$
x_1+x_2+\ldots + x_n \leq K
$$
đúng khi và chỉ khi với $1 \leq i < n$:
$$
\begin{aligned}
&(x_1+x_2+\ldots+x_i \leq k) 
\land\ (x_{i+1}+x_{i+2}+\ldots+x_n \leq k) \\
\land\ \bigwedge_{j=1}^{k} &\bigg( (x_1+x_2+\ldots+x_i \leq k-j) \lor (x_{i+1}+x_{i+2}+\ldots+x_n \leq j-1) \bigg)
\end{aligned}
$$

Nói chung là giống SCAMO, chỉ khác là tính chất 1 sẽ thay đổi dẫn đến bước nối các block cũng thay đổi. Chi tiết có thể xem lại SCAMO.

In [ ]:
from pysat.solvers import Glucose4
import math

class CoreManager:
    def __init__(self):
        self.solver = Glucose4()
        self.clauses = []
        self.vars = {}
        self.vars_count = 0
    
    def add_variable(self, name):
        self.vars_count += 1
        self.vars[name] = self.vars_count
        return self.vars_count
    
    def add_clause(self, clause):
        self.clauses.append(clause)
        self.solver.add_clause(clause)
    
    def solve(self):
        return self.solver.solve()
    
class SCLE:
    def __init__(self, core_manager:CoreManager):
        self.core_manager = core_manager
    
    def add_clause(self, clause:list):
        self.core_manager.add_clause(clause)
    
    def create_subsets(self, variables:list, w:int)->list:
        subsets = [[0]]
        vars = [0] + variables # Add a dummy variable at the beginning of the list to make it 1-indexed
        n = len(vars)-1
        for i in range(1,n+1,w):
            if i+w > n:
                subset = vars[i:]
            else:
                subset = vars[i:i+w]
            subsets.append(subset)
        return subsets

    def create_block(self, block_id:int, subset:list, K:int,w:int,is_AMK:bool=True)->list:
        registers = [[0 for _ in range(K+1)]]
        vars = [0] + subset # Add a dummy variable at the beginning of the list to make it 1-indexed
        w_i = len(vars)-1
        limit = w_i+1 if w_i < w else w
        for j in range(1,limit): # Create the first w_i-1 rows of the block
           register = [0] # Add a dummy variable at the beginning of the row to make it 1-indexed
           for s in range(1,min(j,K)+1): # Create the first min(j,K) columns of the block:
                var_name = f"r_{block_id}_{j}_{s}"
                var_id = self.core_manager.add_variable(var_name)
                register.append(var_id)
           registers.append(register)
       
        # Formula 1
        for j in range(1,w_i):
            self.add_clause([-vars[j],registers[j][1]])
        
        # Formula 2
        for j in range(2,w_i):
            for s in range(1,min(j-1,K)+1):
                self.add_clause([-registers[j-1][s],registers[j][s]])
        
        # Formula 3
        for j in range(2,w_i):
            for s in range(2,min(j,K)+1):
                self.add_clause([-vars[j],-registers[j-1][s-1],registers[j][s]])
        
        # Formula 4
        for j in range(1,min(K,w_i)+1):
            self.add_clause([vars[j], -registers[j][j]])

        # Formula 5
        for j in range(2,w_i):
            for s in range(2,min(j,K)+1):
                self.add_clause([registers[j-1][s-1], -registers[j][s]])
        
        # Formula 6
        for j in range(2,w_i):
            for s in range(1,min(j-1,K)+1):
                self.add_clause([vars[j],registers[j-1][s],-registers[j][s]])
            
        # Formula 7
        if is_AMK:
            for j in range(K+1,w_i+1):
                self.add_clause([-vars[j],-registers[j-1][K]])
        
        return registers

    def create_blocks(self, subsets:list[list],w:int, K:int)->list:
        blocks = [None] # Add a dummy block at the beginning of the list to make it 1-indexed
        
        if len(subsets) == 2:
            blocks.append(self.create_block(1, list(reversed(subsets[1])), K, w, is_AMK=True))
            return blocks

        blocks.append(self.create_block(1, list(reversed(subsets[1])), K, w, is_AMK=True))
        
        for i in range(2,len(subsets)-1):
            LR_subset = subsets[i].copy()
            blocks.append(self.create_block(len(blocks), LR_subset, K, w, is_AMK=True))

            RL_subset = list(reversed(LR_subset))
            blocks.append(self.create_block(len(blocks), RL_subset, K, w, is_AMK=False))

        blocks.append(self.create_block(len(blocks), subsets[-1], K, w, is_AMK=True))
        
        return blocks
    
    def block_to_subset_index(self, block_index:int)->int:
        return (block_index+2)//2
    
    def connect_blocks(self, blocks:list, subsets:list, w:int, K:int):
        M = len(subsets)-1
        blocks_count = len(blocks)-1 # iff 2*(M-1)+1 == blocks_count
        for i in range(1,2*(M-1)+1,2):
            RL_subset_index = self.block_to_subset_index(i)
            LR_subset_index = self.block_to_subset_index(i+1)
            
            w_i_LR = len(subsets[LR_subset_index])
            w_i_RL = len(subsets[RL_subset_index])
            w_i = min(w_i_LR+1,w_i_RL)
            for j in range(2, w_i+1):
                for p in range(1,K+1):
                    left_len = w - j + 1
                    right_len = j - 1

                    left_threshold = K - p + 1
                    right_threshold = p

                    if left_threshold > left_len or right_threshold > right_len:
                        continue

                    self.add_clause([
                        -blocks[i][left_len][left_threshold],
                        -blocks[i+1][right_len][right_threshold]
                    ])

    def amksc(self, variables,w,K):
        if not 1<w<=len(variables):
            raise ValueError("w must be in the range [2,n]")
        
        if not (1<= K < w):
            raise ValueError("K must be in the range [1,w-1]")
        subsets = self.create_subsets(variables,w)
        blocks = self.create_blocks(subsets,w,K)
        self.connect_blocks(blocks, subsets, w, K)
    
    def alksc(self, variables, w, K):
        if not 1 <= K <= w:
            raise ValueError("K must be in the range [1,w]")

        new_K = w - K
        neg_variables = [-v for v in variables]

        if new_K == 0:
            for x in neg_variables:
                self.add_clause([-x])  # tức add_clause([original x])
            return

        self.amksc(neg_variables, w, new_K)

class SCE:
    def __init__(self, core_manager:CoreManager):
        self.core_manager = core_manager
    
    def add_clause(self, clause:list):
        self.core_manager.add_clause(clause)
    
    def new_counter_states(self, variables:list, K:int)->dict:
        registers = {}

        for i in range(len(variables)-1):
            for j in range(min(i+1,K)):
                var_name = f"sce_r_{i}_{j+1}"
                registers[(i,j)] = self.core_manager.add_variable(var_name)

        return registers

    def amk(self, variables:list, K:int)->dict:
        n = len(variables)

        if K < 0:
            self.add_clause([])
            return {}

        if K >= n:
            return {}

        if K == 0:
            for x in variables:
                self.add_clause([-x])
            return {}

        registers = self.new_counter_states(variables,K)

        # Formula 1
        for i in range(n-1):
            self.add_clause([-variables[i],registers[(i,0)]])

        # Formula 2
        for i in range(1,n-1):
            for j in range(min(i,K)):
                self.add_clause([-registers[(i-1,j)],registers[(i,j)]])

        # Formula 3
        for i in range(1,n-1):
            for j in range(1,min(i+1,K)):
                self.add_clause([-variables[i],-registers[(i-1,j-1)],registers[(i,j)]])

        # Formula 8
        for i in range(K,n):
            self.add_clause([-variables[i],-registers[(i-1,K-1)]])

        return registers

    def alk(self, variables:list, K:int)->dict:
        n = len(variables)

        if K <= 0:
            return {}

        if K > n:
            self.add_clause([])
            return {}

        if K == n:
            for x in variables:
                self.add_clause([x])
            return {}

        registers = self.new_counter_states(variables,K)

        # Formula 1
        for i in range(n-1):
            self.add_clause([-variables[i],registers[(i,0)]])

        # Formula 2
        for i in range(1,n-1):
            for j in range(min(i,K)):
                self.add_clause([-registers[(i-1,j)],registers[(i,j)]])

        # Formula 3
        for i in range(1,n-1):
            for j in range(1,min(i+1,K)):
                self.add_clause([-variables[i],-registers[(i-1,j-1)],registers[(i,j)]])

        # Formula 4
        for i in range(1,n-1):
            for j in range(min(i,K)):
                self.add_clause([variables[i],registers[(i-1,j)],-registers[(i,j)]])

        # Formula 5
        for i in range(min(K,n-1)):
            self.add_clause([variables[i],-registers[(i,i)]])

        # Formula 6
        for i in range(1,n-1):
            for j in range(1,min(i+1,K)):
                self.add_clause([registers[(i-1,j-1)],-registers[(i,j)]])

        # Formula 7
        last_x = variables[-1]
        last_prefix = n-2
        if K == 1:
            self.add_clause([registers[(last_prefix,0)],last_x])
        else:
            self.add_clause([registers[(last_prefix,K-1)],last_x])
            self.add_clause([registers[(last_prefix,K-1)],registers[(last_prefix,K-2)]])

        return registers

    def exk(self, variables:list, K:int)->dict:
        return {
            "at_least": self.alk(variables,K),
            "at_most": self.amk(variables,K),
        }

    def range(self, variables:list, u:int, v:int)->dict:
        return {
            "lower": self.alk(variables,u),
            "upper": self.amk(variables,v),
        }


## Cài đặt với bài toán Nurse Rostering Problem (NRP)

Bài toán **Nurse Rostering Problem (NRP)** yêu cầu xây dựng lịch làm việc khả thi cho $n$ y tá trong $d$ ngày. Với mỗi cặp `(y tá, ngày)`, lịch chọn đúng một trong bốn trạng thái:

| Ký hiệu | Ý nghĩa |
|:--:|---|
| $D$ | Ca ngày |
| $E$ | Ca chiều |
| $N$ | Ca đêm |
| $O$ | Nghỉ |

### Biến quyết định

Với $1 \le i \le n$, $1 \le t \le d$ và $s \in \{D,E,N,O\}$, định nghĩa biến Boolean:

$$
x_{i,t,s} =
\begin{cases}
1, & \text{nếu y tá } i \text{ được gán trạng thái } s \text{ ở ngày } t, \\
0, & \text{ngược lại.}
\end{cases}
$$

Trong đó $x_{i,t,O}=1$ nghĩa là y tá $i$ nghỉ ở ngày $t$. Vì mỗi ngày chọn đúng một trạng thái, điều kiện nghỉ tương đương với việc không làm các ca $D,E,N$.

### Ràng buộc cơ sở
1. Nhiều nhất một ca làm việc mỗi ngày
$$
\begin{aligned}
&\bigwedge_{i=1}^{n} \bigwedge_{t=1}^{d} \left( \sum_{s \in \{D,E,N,O\}} x_{i,t,s} =1 \right) \end{aligned}
$$
2. Nhiều nhất 6 ngày làm việc trong mỗi 7 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-6} \left( \sum_{t=j}^{t+6} \neg x_{i,t,O} \leq 6 \right) \text{(Dạng At-Most)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-6} \left( \sum_{t=j}^{t+6} x_{i,t,O} \geq 1 \right) \text{(Dạng At-Least)}
\end{aligned}
$$
Ví dụ biểu diễn

$$
\begin{array}{rcccccccccccccc}
\neg x_{i,1,O} & + & \neg x_{i,2,O} & + & \dots & + & \neg x_{i,7,O} & & & & & & & & \le 6 \\
               &   & \neg x_{i,2,O} & + & \dots & + & \neg x_{i,7,O} & + & \neg x_{i,8,O} & & & & & & \le 6 \\
               &   &                &   & \ddots &   &                &   &                &   & \ddots & & & & \vdots \\
               &   &                &   &        &   & \neg x_{i,d-6,O} & + & \dots          & + & \neg x_{i,d-1,O} & + & \neg x_{i,d,O} & & \le 6
\end{array}
$$

3. Ít nhất 4 ngày nghỉ trong mỗi 14 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13}  x_{i,t,O} \geq 4 \right) \text{(Dạng At-Most)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-6} \left( \sum_{t=j}^{t+6} \neg x_{i,t,O} \leq 10 \right) \text{(Dạng At-Least)}
\end{aligned}
$$

4. Ít nhất 4 ca chiều trong mỗi 14 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} x_{i,t,E} \geq 4 \right) \text{(Dạng At-Least)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} \neg x_{i,t,E} \leq 10 \right) \text{(Dạng At-Most)}
\end{aligned}
$$

5. Nhiều nhất 8 ca chiều trong mỗi 14 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} x_{i,t,E} \leq 8 \right) \text{(Dạng At-Most)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} \neg x_{i,t,E} \geq 6 \right) \text{(Dạng At-Least)}
\end{aligned}
$$

6. Ít nhất 20 ngày làm việc trong mỗi 28 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-27} \left( \sum_{t=j}^{t+27} \neg x_{i,t,O} \geq 20 \right) \text{(Dạng At-Least)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-27} \left( \sum_{t=j}^{t+27} x_{i,t,O} \leq 8 \right) \text{(Dạng At-Most)}
\end{aligned} 
$$
Ngoài ra để tổng quát hơn  thì còn có thể thêm ràng buộc dưới đây thành:
$$
x_{i,t,O} \iff \neg (x_{i,t,D} \lor x_{i,t,E} \lor x_{i,t,N}), \quad 1 \leq i \leq n, 1 \leq t \leq d
$$

7. Nhiều nhất 4 ca đêm trong mỗi 14 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} x_{i,t,N} \leq 4 \right) \text{(Dạng At-Most)}\\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} \neg x_{i,t,N} \geq 10 \right) \text{(Dạng At-Least) }
\end{aligned} 
$$


8. Ít nhất một ca đêm trong mỗi 14 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} x_{i,t,N} \geq 1 \right) \text{(Dạng At-Least)}\\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{t=j}^{t+13} \neg x_{i,t,N} \leq 13 \right) \text{(Dạng At-Most) }
\end{aligned} 
$$

9. Ít nhất 2 ca chiều/đêm trong mỗi 7 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-6} \left( \sum_{t=j}^{t+6} (x_{i,t,E}+x_{i,t,N}) \geq 2 \right) \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{r=2j-1}^{2(j+6)} z_{i,r} \geq 2 \right) \text{(Dạng At-Least)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{r=2j-1}^{2(j+6)} \neg z_{i,r} \leq 12 \right) \text{(Dạng At-Most) }
\end{aligned} 
$$

Với $1 \leq j \leq d-6$ và $2j-1 \leq r \leq 2(j+6)$:
$$
\begin{cases} 
z_{i,r} = x_{i,j,E} &, r = 2j-1 \\ 
z_{i,r} = x_{i,j,N} &, r = 2j 
\end{cases}
$$ 
$\max(r) = 2d$


10. Nhiều nhất 4 ca chiều/đêm trong mỗi 7 ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-6} \left( \sum_{t=j}^{t+6} (x_{i,t,E}+x_{i,t,N}) \leq 4 \right) \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{r=2j-1}^{2(j+6)} z_{i,r} \leq 4 \right) \text{(Dạng At-Least)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-13} \left( \sum_{r=2j-1}^{2(j+6)} \neg z_{i,r} \geq 10 \right) \text{(Dạng At-Most) }
\end{aligned} 
$$


11. Không được làm ca đêm trong hai ngày liên tiếp
$$
\begin{aligned}
&\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-1} \left( \sum_{t=j}^{t+1} x_{i,t,N} \leq 1 \right) \text{(Dạng At-Most)} \\
\iff &\bigwedge_{i=1}^{n}  \bigwedge_{j=1}^{d-1} \left( \sum_{t=j}^{j+1} \neg x_{i,t,N} \geq 1 \right) \text{(Dạng At-Least) }
\end{aligned} 
$$


In [2]:
class NurseRosteringProblem:
    """NRP model whose constraints follow the formulas in Cell 3."""

    SHIFTS = ("D", "E", "N", "O")

    def __init__(self, nurse_count: int, day_count: int):
        if nurse_count <= 0:
            raise ValueError("nurse_count must be positive")
        if day_count <= 0:
            raise ValueError("day_count must be positive")

        self.nurse_count = nurse_count
        self.day_count = day_count
        self.core = CoreManager()
        self.scle = SCLE(self.core)
        self.x = {}
        self._solved = False
        self._is_sat = None

        self._create_decision_variables()

    def _create_decision_variables(self):
        for nurse in range(1, self.nurse_count + 1):
            for day in range(1, self.day_count + 1):
                for shift in self.SHIFTS:
                    name = f"x_{nurse}_{day}_{shift}"
                    self.x[(nurse, day, shift)] = self.core.add_variable(name)

    def shift_var(self, nurse: int, day: int, shift: str) -> int:
        return self.x[(nurse, day, shift)]

    def day_vars(self, nurse: int, day: int) -> list[int]:
        return [self.shift_var(nurse, day, shift) for shift in self.SHIFTS]

    def shift_sequence(self, nurse: int, shift: str) -> list[int]:
        return [
            self.shift_var(nurse, day, shift)
            for day in range(1, self.day_count + 1)
        ]

    def evening_night_window(self, nurse: int, start_day: int, width: int = 7) -> list[int]:
        literals = []
        for day in range(start_day, start_day + width):
            literals.append(self.shift_var(nurse, day, "E"))
            literals.append(self.shift_var(nurse, day, "N"))
        return literals

    def shift_coverage(self, day: int, shift: str) -> list[int]:
        return [
            self.shift_var(nurse, day, shift)
            for nurse in range(1, self.nurse_count + 1)
        ]

    def add_at_least(self, literals: list[int], lower: int):
        n = len(literals)
        if lower <= 0:
            return
        if lower > n:
            self.core.add_clause([])
            return
        if lower == n:
            for literal in literals:
                self.core.add_clause([literal])
            return
        self.scle.alksc(literals, n, lower)

    def add_at_most(self, literals: list[int], upper: int):
        n = len(literals)
        if upper < 0:
            self.core.add_clause([])
            return
        if upper >= n:
            return
        if upper == 0:
            for literal in literals:
                self.core.add_clause([-literal])
            return
        self.scle.amksc(literals, n, upper)

    def add_exactly(self, literals: list[int], count: int):
        self.add_at_least(literals, count)
        self.add_at_most(literals, count)

    def add_scle_at_least(self, literals: list[int], width: int, lower: int):
        if len(literals) >= width:
            self.scle.alksc(literals, width, lower)

    def add_scle_at_most(self, literals: list[int], width: int, upper: int):
        if len(literals) >= width:
            self.scle.amksc(literals, width, upper)

    def add_daily_coverage(self, min_demand=None, max_demand=None):
        min_demand = min_demand or {}
        max_demand = max_demand or {}

        for day in range(1, self.day_count + 1):
            for shift, lower in min_demand.items():
                self.add_at_least(self.shift_coverage(day, shift), lower)
            for shift, upper in max_demand.items():
                self.add_at_most(self.shift_coverage(day, shift), upper)

    def add_constraints_for_nurse(self, nurse: int):
        off = self.shift_sequence(nurse, "O")
        evening = self.shift_sequence(nurse, "E")
        night = self.shift_sequence(nurse, "N")

        # (1) sum_{s in {D,E,N,O}} x_{i,t,s} = 1.
        for day in range(1, self.day_count + 1):
            self.add_exactly(self.day_vars(nurse, day), 1)

        # (2) sum_{t=j}^{j+6} not O <= 6 <=> sum O >= 1.
        self.add_scle_at_least(off, 7, 1)

        # (3) sum_{t=j}^{j+13} O >= 4.
        self.add_scle_at_least(off, 14, 4)

        # (4) sum_{t=j}^{j+13} E >= 4.
        self.add_scle_at_least(evening, 14, 4)

        # (5) sum_{t=j}^{j+13} E <= 8.
        self.add_scle_at_most(evening, 14, 8)

        # (6) sum working >= 20 <=> sum O <= 8 over every 28 days.
        self.add_scle_at_most(off, 28, 8)

        # (7) sum_{t=j}^{j+13} N <= 4.
        self.add_scle_at_most(night, 14, 4)

        # (8) sum_{t=j}^{j+13} N >= 1.
        self.add_scle_at_least(night, 14, 1)

        en_literals = self.evening_night_window(nurse, 1, self.day_count)
        # (9) sum_{t=j}^{j+6} E/N >= 2.
        self.add_scle_at_least(en_literals, 14, 2)

        # (10) sum_{t=j}^{j+6} E/N <= 4.
        self.add_scle_at_most(en_literals, 14, 4)

        # (11) sum_{t=j}^{j+1} N <= 1.
        self.add_scle_at_most(night, 2, 1)

    def add_constraints(self, min_demand=None, max_demand=None):
        for nurse in range(1, self.nurse_count + 1):
            self.add_constraints_for_nurse(nurse)

        # Optional coverage constraints are not part of Cell 3, but can be added for demand.
        self.add_daily_coverage(min_demand=min_demand, max_demand=max_demand)
        return self

    def solve(self) -> bool:
        if not self._solved:
            self._is_sat = self.core.solve()
            self._solved = True
        return self._is_sat

    def model(self) -> list[int] | None:
        if not self.solve():
            return None
        return self.core.solver.get_model()

    def schedule(self) -> list[list[str]] | None:
        model = self.model()
        if model is None:
            return None

        positive = {literal for literal in model if literal > 0}
        schedule = []
        for nurse in range(1, self.nurse_count + 1):
            row = []
            for day in range(1, self.day_count + 1):
                row.append(next(
                    shift for shift in self.SHIFTS
                    if self.shift_var(nurse, day, shift) in positive
                ))
            schedule.append(row)
        return schedule


def solve_nrp(nurse_count=10, day_count=28, min_demand=None, max_demand=None):
    nrp = NurseRosteringProblem(nurse_count, day_count).add_constraints(
        min_demand=min_demand,
        max_demand=max_demand,
    )
    return nrp, nrp.schedule()


def count_windows(row: list[str], width: int, predicate) -> list[int]:
    return [
        sum(1 for shift in row[start:start + width] if predicate(shift))
        for start in range(len(row) - width + 1)
    ]


def validate_nrp_schedule(schedule, day_count, min_demand=None, max_demand=None):
    checks = []
    min_demand = min_demand or {}
    max_demand = max_demand or {}

    for nurse_id, row in enumerate(schedule, start=1):
        checks.append((
            f"Nurse {nurse_id}: (1) exactly one status per day",
            len(row) == day_count and all(shift in NurseRosteringProblem.SHIFTS for shift in row),
        ))

        if day_count >= 7:
            checks.append((f"Nurse {nurse_id}: (2) >= 1 O / 7", all(v >= 1 for v in count_windows(row, 7, lambda s: s == "O"))))
            checks.append((f"Nurse {nurse_id}: (9) >= 2 E/N / 7", all(v >= 2 for v in count_windows(row, 7, lambda s: s in {"E", "N"}))))
            checks.append((f"Nurse {nurse_id}: (10) <= 4 E/N / 7", all(v <= 4 for v in count_windows(row, 7, lambda s: s in {"E", "N"}))))

        if day_count >= 14:
            checks.append((f"Nurse {nurse_id}: (3) >= 4 O / 14", all(v >= 4 for v in count_windows(row, 14, lambda s: s == "O"))))
            checks.append((f"Nurse {nurse_id}: (4) >= 4 E / 14", all(v >= 4 for v in count_windows(row, 14, lambda s: s == "E"))))
            checks.append((f"Nurse {nurse_id}: (5) <= 8 E / 14", all(v <= 8 for v in count_windows(row, 14, lambda s: s == "E"))))
            checks.append((f"Nurse {nurse_id}: (7) <= 4 N / 14", all(v <= 4 for v in count_windows(row, 14, lambda s: s == "N"))))
            checks.append((f"Nurse {nurse_id}: (8) >= 1 N / 14", all(v >= 1 for v in count_windows(row, 14, lambda s: s == "N"))))

        if day_count >= 28:
            checks.append((f"Nurse {nurse_id}: (6) >= 20 working / 28", all(v >= 20 for v in count_windows(row, 28, lambda s: s != "O"))))

        checks.append((
            f"Nurse {nurse_id}: (11) no consecutive N",
            all(not (row[d] == "N" and row[d + 1] == "N") for d in range(len(row) - 1)),
        ))

    for day in range(day_count):
        for shift, lower in min_demand.items():
            count = sum(row[day] == shift for row in schedule)
            checks.append((f"Day {day + 1}: >= {lower} {shift}", count >= lower))
        for shift, upper in max_demand.items():
            count = sum(row[day] == shift for row in schedule)
            checks.append((f"Day {day + 1}: <= {upper} {shift}", count <= upper))

    return checks


nrp, schedule = solve_nrp(nurse_count=30, day_count=84)

print("SAT:", schedule is not None)
print("Variables:", nrp.core.vars_count)
print("Clauses:", len(nrp.core.clauses))

if schedule is not None:
    for nurse_id, row in enumerate(schedule, start=1):
        print(f"Nurse {nurse_id}:", " ".join(row))

    validation = validate_nrp_schedule(schedule, nrp.day_count)
    failed = [name for name, ok in validation if not ok]
    print("Validation checks:", len(validation))
    print("Failed:", len(failed))
    print("All constraints satisfied:", len(failed) == 0)
    for name in failed:
        print("FAIL -", name)


SAT: True
Variables: 276300
Clauses: 1045320
Nurse 1: E N E D O O O D E E D D N O E N E D O O O E E N D D N O E E D N O O O E E D E N D O E E D N O O O D E E E D E O D E D N O O O D E D E E D O E N D E O O O E N D N E D O
Nurse 2: O N D O E D O E D D N E O E O E D O N D O E D D E E O E O D E O N E O N D D D E O E O E D O E N O N D D E E O N O E D O E N O E D E D E O D O N E O D D O E E E E D O D
Nurse 3: E E O O O D E E N O D N D D E E O O O D E E D O N D E D E E O O O D E N D O E D E E D N O O O E N D E O D D E E N D O O O E E D D O E N E N D D O O O E E E D O D N E E
Nurse 4: E E E O O O N E E D E D O N D N E O O O D E E E E D O D N E N O O O E D D E E D O E N D N O O O E E E D D N O E E E D O O O N E D E D E O D E N E O O O D N E N E D O D
Nurse 5: O O D N O D E E D E O E D E O O N E O E E D D N O N E N O O D E O E E N D D O E N D O O E E O D E E D E O N D E O O D N O E E E D D O E E D O O E N O D D E E E O E D D
Nurse 6: E D D D N E O E O O D N E O E D D D E E O D O O E N E O D D D